In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "async-geotiff>=0.4",
#     "lazycogs",
#     "lonboard>=0.15.0",
#     "numpy>=2",
#     "obstore>=0.9.2",
#     "pillow>=12.1.1",
#     "rustac",
#     "sidecar>=0.8.1",
# ]
# ///

# CDL across many tiles, reprojected, in Lonboard — via `lazycogs`

The companion notebook (`raster-cog-cdl-pc.ipynb`) renders one Planetary Computer CDL tile interactively in its native Albers CRS, with deck.gl-raster reprojecting in the browser.

This one takes the other path: use [`lazycogs`] to mosaic _many_ MPC CDL tiles into a single reprojected xarray DataArray (Web Mercator), then drop it onto a map as a `BitmapLayer`. Same stack underneath — [`rustac`] for STAC, [`async-geotiff`] for COG I/O, [`obstore`] for cloud storage — but the lazy-xarray path means we can cover anything from a county to CONUS without giving up the no-server, no-GDAL story.

[`lazycogs`]: https://github.com/developmentseed/lazycogs
[`rustac`]: https://github.com/stac-utils/rustac-py
[`async-geotiff`]: https://developmentseed.org/async-geotiff/
[`obstore`]: https://developmentseed.org/obstore/

## Run

```
uvx juv run cdl-lazycogs-mosaic.ipynb
```

Optional: set `PC_SDK_SUBSCRIPTION_KEY` for higher MPC rate limits.

In [ ]:
import base64
import io
from urllib.parse import urlparse

import lazycogs
import numpy as np
import rustac
from async_geotiff import GeoTIFF
from obstore.auth.planetary_computer import PlanetaryComputerCredentialProvider
from obstore.store import AzureStore
from PIL import Image
from pyproj import Transformer
from sidecar import Sidecar

from lonboard import Map
from lonboard.layer import BitmapLayer

## Pick a region

Bounds in Web Mercator (EPSG:3857). Iowa here — change freely. The size of this bbox × `resolution` determines how much pixel data we materialize.

In [ ]:
DST_CRS = "EPSG:3857"
RESOLUTION = 500.0  # meters/pixel in Mercator; raise to 1000 for CONUS, drop to 100 for county

# Iowa-ish bbox in Web Mercator
to_3857 = Transformer.from_crs("EPSG:4326", DST_CRS, always_xy=True)
DST_BBOX = (
    *to_3857.transform(-96.7, 40.4),
    *to_3857.transform(-90.1, 43.6),
)

# Convert to lon/lat for the STAC search
to_4326 = Transformer.from_crs(DST_CRS, "EPSG:4326", always_xy=True)
BBOX_4326 = to_4326.transform_bounds(*DST_BBOX)
DST_BBOX, BBOX_4326

## Cache STAC items to a local geoparquet via `rustac`

`lazycogs.open()` reads from a geoparquet file (DuckDB-backed). `rustac.search_to` writes one in one call.

In [ ]:
PARQUET = "cdl_items.parquet"

await rustac.search_to(
    PARQUET,
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    collections=["usda-cdl"],
    bbox=list(BBOX_4326),
    datetime="2021",
    filter={"op": "=", "args": [{"property": "usda_cdl:type"}, "cropland"]},
)
PARQUET

## Configure the obstore + PC credential provider

All CDL COGs live under one Azure container, so one `AzureStore` + one credential provider covers every item. `path_from_href` tells lazycogs how to map a STAC asset href to a path _inside_ that store.

In [ ]:
store = AzureStore(
    credential_provider=PlanetaryComputerCredentialProvider(
        url="https://landcoverdata.blob.core.windows.net/usda-cdl/",
    ),
)


def path_from_href(href: str) -> str:
    # https://landcoverdata.blob.core.windows.net/usda-cdl/tiles/.../foo.tif
    # ->                                                       tiles/.../foo.tif
    return urlparse(href).path.split("/", 2)[2]

## Open the mosaic as a lazy xarray DataArray

lazycogs builds the output grid in our target CRS + resolution, identifies which COGs intersect each chunk, fetches only the byte ranges needed, and reprojects with `pyproj` + numpy. Nothing is read until we `.compute()` (or `.values`).

In [ ]:
da = lazycogs.open(
    PARQUET,
    bbox=DST_BBOX,
    crs=DST_CRS,
    resolution=RESOLUTION,
    bands=["cropland"],
    dtype="uint8",
    nodata=0,
    store=store,
    path_from_href=path_from_href,
)
da

## Borrow the CDL colormap from one COG

The cropland COGs embed the USDA-CDL color table. Open one item directly with `async-geotiff` to extract it — we only need the LUT, not the pixels.

In [ ]:
sample_href = rustac.DuckdbClient().search(PARQUET, max_items=1)[0]["assets"]["cropland"]["href"]
sample_geotiff = await GeoTIFF.open(path_from_href(sample_href), store=store)
cmap_array = sample_geotiff.colormap.as_array()  # (256, 3) uint8
cmap_array.shape

## Materialize, colorize, encode

`.values` triggers all the byte-range fetches and reprojection. For ~3000×4000 px at 500 m resolution over Iowa, this is small — for CONUS, bump `resolution` to 1000+ or use `chunks=...` to get a dask-backed array and `.persist()`.

In [ ]:
arr = da.isel(band=0, time=0).values  # (y, x) uint8

alpha = np.where(arr == 0, 0, 255).astype(np.uint8)
rgba = np.empty((*arr.shape, 4), dtype=np.uint8)
rgba[..., :3] = cmap_array[arr]
rgba[..., 3] = alpha

buf = io.BytesIO()
Image.fromarray(rgba, mode="RGBA").save(buf, format="PNG")
data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
len(buf.getvalue()), arr.shape

## Display

`BitmapLayer` takes `bounds` as `[west, south, east, north]` in lon/lat — deck.gl handles the Mercator stretch. Because we rendered the image _in_ Web Mercator, the stretch lines up correctly.

In [ ]:
west, south, east, north = BBOX_4326
layer = BitmapLayer(image=data_url, bounds=[west, south, east, north])

sidecar = Sidecar(anchor="split-right")
m = Map(layer, height=800)
with sidecar:
    display(m)

## Scaling notes

- **County** (~50 km): `RESOLUTION = 100.0` — full 30 m gets you ~25 MP, fine in memory.
- **State**: `RESOLUTION = 500.0` — what's set here, ~12 MP over Iowa.
- **CONUS**: `RESOLUTION = 2000.0`, and switch to dask: `da = lazycogs.open(..., chunks={"y": 2048, "x": 2048})` then `arr = da.isel(band=0, time=0).compute().values`. Or skip the single-`BitmapLayer` path entirely and tile it.
- **Multi-year change detection**: drop the `datetime="2021"` filter; lazycogs returns a `(band, time, y, x)` cube you can diff.